In [1]:
import numpy as np
import pandas as pd
import time
import math
import os

from numba import cuda
import kagglehub

In [2]:
path = kagglehub.dataset_download("erikbiswas/higgs-uci-dataset")

print("Path to dataset files:", path)

100%|██████████| 2.70G/2.70G [00:26<00:00, 108MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/erikbiswas/higgs-uci-dataset/versions/1


In [3]:
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/root/.cache/kagglehub/datasets/erikbiswas/higgs-uci-dataset/versions/1/HIGGS.csv


In [4]:
csv_file = os.path.join(path, "HIGGS.csv")

df = pd.read_csv(
    csv_file,
    header=None,
    nrows=100000
)

print(df.shape)

(100000, 29)


In [5]:
features = df.iloc[:, 1:].astype(np.float32)

print(features.shape)

(100000, 28)


In [6]:
data = features.to_numpy().ravel()

print("Total Values:", len(data))
print("Memory:", data.nbytes / 1024**2, "MB")

Total Values: 2800000
Memory: 10.68115234375 MB


In [7]:
@cuda.jit
def sigmoid_kernel(inp, out):

    idx = cuda.grid(1)

    if idx < inp.size:

        x = inp[idx]

        out[idx] = (
            1.0 /
            (1.0 + math.exp(-x))
        )

In [8]:
def cpu_sigmoid(x):

    return (
        1.0 /
        (1.0 + np.exp(-x))
    )

In [9]:
start_cpu = time.time()

cpu_output = cpu_sigmoid(data)

cpu_time = time.time() - start_cpu

print("CPU Time:", cpu_time)

CPU Time: 0.0227663516998291


In [10]:
gpu_total_start = time.time()

d_input = cuda.to_device(data)

d_output = cuda.device_array_like(data)

In [11]:
threads_per_block = 256

blocks_per_grid = (
    len(data)
    + threads_per_block
    - 1
) // threads_per_block

print(blocks_per_grid)

10938


In [12]:
kernel_start = time.time()

sigmoid_kernel[
    blocks_per_grid,
    threads_per_block
](
    d_input,
    d_output
)

cuda.synchronize()

kernel_time = (
    time.time()
    - kernel_start
)

print("GPU Kernel Time:", kernel_time)

GPU Kernel Time: 1.9757800102233887


In [13]:
gpu_output = d_output.copy_to_host()

gpu_total_time = (
    time.time()
    - gpu_total_start
)

print("GPU Total Time:", gpu_total_time)

GPU Total Time: 2.3922278881073


In [14]:
correct = np.allclose(
    cpu_output,
    gpu_output,
    atol=1e-5
)

print("Results Match:", correct)

Results Match: True


In [15]:
print("\n===== Performance Comparison =====")

print(f"CPU Time : {cpu_time:.6f} sec")

print(f"GPU Kernel Time : {kernel_time:.6f} sec")

print(f"GPU Total Time : {gpu_total_time:.6f} sec")

print(
    f"Kernel Speedup : "
    f"{cpu_time/kernel_time:.2f}x"
)

print(
    f"Overall Speedup : "
    f"{cpu_time/gpu_total_time:.2f}x"
)


===== Performance Comparison =====
CPU Time : 0.022766 sec
GPU Kernel Time : 1.975780 sec
GPU Total Time : 2.392228 sec
Kernel Speedup : 0.01x
Overall Speedup : 0.01x


The GPU implementation initially appeared slower than the CPU implementation because the benchmark included CUDA kernel compilation overhead and data transfer costs. Additionally, the selected subset of the HIGGS dataset was relatively small, allowing the highly optimized NumPy CPU implementation to outperform the GPU. For larger datasets, the parallel execution capabilities of CUDA become more effective and can provide significant acceleration.
